# Optimización de Crop4

Se maximiza el SSIM medio sobre las mismas mezclas de entrenamiento y con la misma semilla en cada prueba. La mezcla usa pesos iguales.


In [11]:
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf
import optuna

project_root = next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "project" / "CROP" / "models").is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from project.loader import Loader
from project.CROP.models.crop4 import Crop4


#SEED = 58642
SEED = 3333

DATASET = "fashion"
INTER= 256
LAT=64 
CVAE_NAME = f"int_{INTER}_lat_{LAT}_{DATASET}.keras"
PREDICTOR_NAME = f"CCE_Conv2D_{DATASET}.keras"
N_IMAGES = 1000
N_TRIALS = 100
TARGET_METRIC = "ssim"  # Maximizar el SSIM medio de las estimaciones.
SEPARATION_ITERATIONS = 30
assert TARGET_METRIC in {"acc_both", "ssim", "recon_bpsnr"}

FIXED_PARAMS = {
    "alpha_1": -2,
    "alpha_2": -22,
    "bias": 0.22,
    "slope": 22,
    "alpha_mix": 0.5,
}

tf.keras.utils.set_random_seed(SEED)
data = Loader.data(dataset=DATASET)
cvae = Loader.cvae(model_name=CVAE_NAME)
predictor = Loader.predictor(model_name=PREDICTOR_NAME)

# Generador independiente de las semillas del CVAE y de Optuna.
rng = np.random.default_rng(SEED)
idx0 = rng.permutation(len(data["x_train"]))[:N_IMAGES]
idx1 = rng.permutation(len(data["x_train_1"]))[:N_IMAGES]
train_batch = (
    data["x_train"][idx0], data["x_train_1"][idx1],
    data["y_train"][idx0], data["y_train_1"][idx1],
)


def scalar_metric(metrics, key):
    value = metrics[key]
    # Las métricas de calidad devuelven (media, desvío).
    if isinstance(value, (tuple, list)):
        value = value[0]
    return float(value)


def objective(trial):
    params = {
        **FIXED_PARAMS,
        "delta": trial.suggest_float("delta", 0.0, 1.0),
        "confidence_scale": trial.suggest_float("confidence_scale", 0.01, 1.0, log=True),
        "confidence_floor": trial.suggest_float("confidence_floor", 0.01, 1.0, log=True),
    }
    tf.keras.utils.set_random_seed(SEED)
    model = Crop4(cvae=cvae, predictor=predictor)
    metrics = model.unmix(
        *train_batch,
        params=params,
        iterations=SEPARATION_ITERATIONS,
        labels=data["labels"],
        show_image=False,
        show_metrics=False,
    )
    score = scalar_metric(metrics, TARGET_METRIC)
    if not np.isfinite(score):
        raise ValueError(f"Métrica no finita: {TARGET_METRIC}={score}")
    trial.set_user_attr("effective_params", params)
    for key in ("acc_both", "ssim", "recon_bpsnr"):
        trial.set_user_attr(key, scalar_metric(metrics, key))
    return score


Usando fashion como dataset


## Estudio

Se ajusta `delta` entre 0 y 1 en escala lineal. Se ajustan `confidence_scale` y `confidence_floor` entre 0.01 y 1 en escala logarítmica. Cada prueba usa 30 iteraciones de separación y se guarda en SQLite.


In [12]:
from datetime import datetime

output_dir = project_root / "project" / "experiments" / "optim_params" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
study_path = output_dir / f"crop4_{DATASET}_{TARGET_METRIC}_{run_id}.sqlite3"
study = optuna.create_study(
    direction="maximize",
    storage=f"sqlite:///{study_path}",
    study_name=f"Crop4_{DATASET}_{TARGET_METRIC}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
print("Estudio persistente:", study_path, flush=True)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = {**FIXED_PARAMS, **study.best_params}
print("Mejor valor:", study.best_value)
print("Parámetros completos:", best_params)


[I 2026-11-24 01:29:35,082] A new study created in RDB with name: Crop4_fashion_ssim


Estudio persistente: /home/galsan/Desktop/tesis/CROP_1/project/experiments/optim_params/results/crop4_fashion_ssim_20261124_012934.sqlite3


[I 2026-11-24 01:29:38,903] Trial 0 finished with value: 0.7257802486419678 and parameters: {'delta': 0.7515756072044787, 'confidence_scale': 0.01653874498578343, 'confidence_floor': 0.09380857915942345}. Best is trial 0 with value: 0.7257802486419678.
[I 2026-11-24 01:29:42,770] Trial 1 finished with value: 0.7146751880645752 and parameters: {'delta': 0.4998311777480342, 'confidence_scale': 0.20616740190582394, 'confidence_floor': 0.02963257706116777}. Best is trial 0 with value: 0.7257802486419678.
[I 2026-11-24 01:29:46,701] Trial 2 finished with value: 0.7148727774620056 and parameters: {'delta': 0.6127948906605941, 'confidence_scale': 0.017350212840849595, 'confidence_floor': 0.26570797725119155}. Best is trial 0 with value: 0.7257802486419678.
[I 2026-11-24 01:29:50,708] Trial 3 finished with value: 0.7060317397117615 and parameters: {'delta': 0.37256054205032785, 'confidence_scale': 0.11208217387727122, 'confidence_floor': 0.033618177569389104}. Best is trial 0 with value: 0.725

Mejor valor: 0.7392407059669495
Parámetros completos: {'alpha_1': -2, 'alpha_2': -22, 'bias': 0.22, 'slope': 22, 'alpha_mix': 0.5, 'delta': 0.7484575455399347, 'confidence_scale': 0.016679948674418854, 'confidence_floor': 0.010314311115229178}


## Visualización de la optimización

Gráficos con Matplotlib: no requieren Plotly ni instalaciones dentro del notebook.

In [ ]:
from optuna.visualization.matplotlib import plot_slice

plot_slice(study, params=list(study.best_params), target_name=TARGET_METRIC)
